# Introduction to `numpy`: Part 2 

See [Part 1](02_02_Modules_in_python_numpy_Part1.ipynb) for array creation, dtypes, and the OOP vocabulary (instance/attribute/method). Here we cover what makes arrays actually *useful* (i.e. speed and indexing) plus reading data from a file.

## Table of Content

- [II.2 Array copies and views](#II.2)
- [II.3 Shape manipulation](#II.3)
- [II.4 What makes numpy arrays fast?](#II.4)
    * ufunc, aggregation, broadcasting, indexing/masking/fancy indexing
- [II.5 Reading arrays from a file & string formatting](#II.5)
- [II.6 A few more useful functions](#II.6)
- [II.7 Summary](#II.7)
- [References](#refs)


## II.2 Array copies and views <a class="anchor" id="II.2"></a>

`b = a` does **not** create a new array -- `b` is just another name for the same object (exactly like aliasing a mutable list). Two dedicated ways to get an actual new array:

- `b = a.copy()`: a fully independent copy, in new memory. Modifying `b` never affects `a`.
- `b = a.view()`: a *new object* that shares the same underlying data as `a`. Modifying `b`'s elements *does* affect `a` (and vice versa). 

In [1]:
import numpy as np
a = np.array([1, 2, 3, 4])
b_copy = a.copy()
b_view = a.view()

b_copy[0] = 99
b_view[1] = 77

print('a       :', a)          # only b_view's change is reflected in a
print('b_copy  :', b_copy)
print('b_view  :', b_view)


a       : [ 1 77  3  4]
b_copy  : [99  2  3  4]
b_view  : [ 1 77  3  4]


**Quick exercise.** Create an array `a = np.arange(5)`. Set `b = a` (plain assignment, not `.copy()`/`.view()`), then modify an element of `b`. Is `a` affected? Explain in one line why, based on what you just saw above.


In [2]:
# Quick exercise: plain assignment vs. copy/view
a = np.arange(5)
print('Original a: ', a)
b = a 
b[3] = 19999
print('Array a after modifying b: a = ', a)

Original a:  [0 1 2 3 4]
Array a after modifying b: a =  [    0     1     2 19999     4]


You may wonder why one would like to use a view. There are effectively cases where a view can be useful:
- A *slicing* operation creates a **view** on the original array. It can be useful to modify a corrupted part of an image (cast into a 2D numpy array), by just dealing with the slice. This will then also modify the original (larger image).
- If you are only interested in a specific part of the data (i.e. part of your numpy array), and do not plan to use any more the original data. You can slice it (which creates a view, i.e. not allocating more memory), and just continue manipulating the view.

## II.3 Shape manipulation <a class="anchor" id="II.3"></a>

**Reshaping** (`a.reshape(new_shape)`) reorganizes the elements of an array into a different shape, without changing the total number of elements. Use `-1` for one dimension to have it inferred automatically.


In [3]:
a = np.array([[1, 2, 3], [4, 5, 6]])
print(a.shape)
b = a.reshape((3, 2))
print(b)

c = a.reshape((3, -1))    # -1 -> "figure this dimension out for me"
print(c.shape)


(2, 3)
[[1 2]
 [3 4]
 [5 6]]
(3, 2)


**Flattening** (`a.flatten()`) collapses all dimensions into a single 1D array. Adding a new axis can be done with `np.newaxis` (e.g. `a[:, :, np.newaxis]` adds a 3rd dimension) -- handy for broadcasting (next section), less commonly needed otherwise.


In [4]:
a3d = np.array([[[1, 2, 3], [4, 5, 6]], [[4, 5, 6], [-1, -2, -3]]])
print(a3d.shape)
print(a3d.flatten())


(2, 2, 3)
[ 1  2  3  4  5  6  4  5  6 -1 -2 -3]


**Quick exercise.** Create an array of values 0 to 99 (`np.arange(100)`) and reshape it into shape `(4, 5, 5)`. Check the result's `.shape`. Then flatten it back to 1D and confirm you recover the original values.


In [5]:
# Quick exercise: reshape (4, 5, 5) and flatten back
a100 = np.arange(100)
a100_reshaped = a100.reshape(4, 5, 5 )
print(np.shape(a100_reshaped))


(4, 5, 5)


## II.4 What makes `numpy` arrays fast? <a class="anchor" id="II.4"></a>

Python is fast to *write*, but slow to *execute*, especially in `for` loops -- every iteration re-checks the type of each value. `numpy` sidesteps this with four strategies: **ufuncs**, **aggregation**, **broadcasting**, and **indexing/masking/fancy indexing**.

### `ufunc`s: elementwise operations

A `ufunc` (universal function) is a fast, compiled, **elementwise** operation: all arithmetic (`+ - * / **`), math functions (`sin`, `exp`, `log10`, ...), and comparisons (`< > == ...`).


In [6]:
a = [1, 2, 3, 4, 5]
b_loop = [val + 5 for val in a]     # python list comprehension
print(b_loop)

a_np = np.array(a)
b_np = a_np + 5                      # numpy: cleaner, and much faster at scale
print(b_np)


[6, 7, 8, 9, 10]
[ 6  7  8  9 10]


**Exercise.** Implement the same "+5" operation for a list/array of 1000 elements, both ways, and use `%timeit` on each to see the speed difference.


In [7]:
# Exercise: %timeit list comprehension vs. numpy, 1000 elements
%timeit [val + 5 for val in a]

102 ns ± 1.28 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [8]:
%timeit b_np = a_np + 5   

450 ns ± 6.05 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [9]:
# But if you increase the size of the list: 
a = list(range(1000))
%timeit [val + 5 for val in a]

17 μs ± 333 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [10]:
a_np = np.array(a)
b_np = a_np + 5                      

In [11]:
%timeit a_np + 5 

682 ns ± 19.8 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


**Exercise.** Given an array of angles in degrees, use `ufunc`s to (a) convert them to radians (`np.radians()`), and (b) compute the sine, cosine and tangent of each.


In [12]:
angles_deg = np.array([0, 30, 45, 60, 90])

# (a) convert to radians, (b) sin/cos/tan
# (a) convert to radians 
angles_rad = np.radians(angles_deg)
angles_rad


array([0.        , 0.52359878, 0.78539816, 1.04719755, 1.57079633])

In [13]:
# (b) sine, cosine, tangent 
tan_all = np.tan(angles_rad)
cos_all = np.cos(angles_rad)
sin_all = np.sin(angles_rad)
for i, angle in enumerate(angles_deg):
    print(f'angle {angle}: cos={cos_all[i]:.3f} ; sin = {sin_all[i]:.3f}; tan={tan_all[i]:.3f}')

angle 0: cos=1.000 ; sin = 0.000; tan=0.000
angle 30: cos=0.866 ; sin = 0.500; tan=0.577
angle 45: cos=0.707 ; sin = 0.707; tan=1.000
angle 60: cos=0.500 ; sin = 0.866; tan=1.732
angle 90: cos=0.000 ; sin = 1.000; tan=16331239353195370.000


### Aggregation: summarizing an array

Functions that reduce an array to a single (or per-axis) summary value: `.min()`, `.max()`, `.sum()`, `.mean()`, `.std()`, `.argmin()`, `.argmax()`, ... On a multi-dimensional array, `axis=0` aggregates down the rows (column-wise result), `axis=1` aggregates across columns (row-wise result).


In [14]:
M = np.random.randint(0, 10, (10, 4))
print(M)
print('Sum over rows -for each column (4 columns)', M.sum(axis=0))   # one value per column
print('Sum over columns for each row (10 rows)', M.sum(axis=1))   # one value per row


[[5 2 0 6]
 [9 0 0 8]
 [1 8 5 5]
 [8 2 4 8]
 [5 9 1 5]
 [5 4 3 8]
 [8 1 7 6]
 [9 8 8 4]
 [0 3 3 6]
 [7 8 3 1]]
Sum over rows -for each column (4 columns) [57 45 34 57]
Sum over columns for each row (10 rows) [13 17 19 22 20 20 22 29 12 19]


**Exercise.** Given `arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])`:
- Compute the mean, the max, and the min.
- Compute the cumulative sum and cumulative product, along rows and along columns.
- **Bonus:** find the index of the maximum value with `arr.argmax()`. Why is it a single number rather than a (row, column) pair? (Hint: check `np.unravel_index`.)

In [15]:
# Exercise: aggregation on a 2D array
arr = np.array([[1, 2, 3], [4, 5, 9], [7, 8, 0]])
print(arr)
print('--------------')
print(f'mean: {arr.mean()}, max: {arr.mean()}, min {arr.min()}')
print(f'Cumulative sum: {arr.cumsum()}')
print(f'Index of maximum', arr.argmax())
print('that is a single index because python flatten the array ...')
print('I can get the 2D index with ', np.unravel_index(arr.argmax(), shape=np.shape(arr))) 

[[1 2 3]
 [4 5 9]
 [7 8 0]]
--------------
mean: 4.333333333333333, max: 4.333333333333333, min 0
Cumulative sum: [ 1  3  6 10 15 24 31 39 39]
Index of maximum 5
that is a single index because python flatten the array ...
I can get the 2D index with  (np.int64(1), np.int64(2))


### Broadcasting

Broadcasting lets `numpy` apply a `ufunc` to two arrays of *different* shapes, by virtually stretching the smaller one, without actually copying data. Starting from the trailing (rightmost) dimension, two dimensions are compatible if they're equal, or if one of them is `1`.


In [16]:
row = np.arange(3)              # shape (3,)
col = np.arange(3).reshape(3, 1) # shape (3, 1)
print(col + row)                # broadcasts to shape (3, 3)


[[0 1 2]
 [1 2 3]
 [2 3 4]]


A very common physics/astro use of broadcasting: normalizing each row of a matrix by its own mean, with no loop at all.


In [17]:
M = np.array([[1., 2., 3.], [10., 20., 30.]])
row_means = M.mean(axis=1)             # shape (2,)
M_normalized = M / row_means[:, np.newaxis]   # reshape to (2, 1) so it broadcasts against (2, 3)
print(M_normalized)


[[0.5 1.  1.5]
 [0.5 1.  1.5]]


In [18]:
import numpy as np
M = np.array([[1., 2., 3.], [10., 20., 30.]])
print('Original array of shape', M.shape)
print(M) 

row_means = M.mean(axis=1)             # shape (2,)
print('Mean of each row')
print(row_means)

print('Normalising each row by its mean (needs reshaping to (2, 1) so it broadcasts against (2, 3))')
M_normalized = M / row_means[:, np.newaxis]   # reshape to (2, 1) so it broadcasts against (2, 3)
print(M_normalized)

Original array of shape (2, 3)
[[ 1.  2.  3.]
 [10. 20. 30.]]
Mean of each row
[ 2. 20.]
Normalising each row by its mean (needs reshaping to (2, 1) so it broadcasts against (2, 3))
[[0.5 1.  1.5]
 [0.5 1.  1.5]]


**General rule**

When operating on two arrays, NumPy compares their shapes element-wise. It starts with the trailing (i.e. rightmost) dimension and works its way left. Two dimensions are compatible when
- they are equal, or
- one of them is 1.

If these conditions are not met, a ValueError: operands could not be broadcast together exception is thrown.

**Application to 3 cases:** 

![From astroML book](../Figures/fig_broadcast_visual_1.png)


### Slicing, masking, and fancy indexing

Recall slicing: `a[start:stop:step]`, with negative steps counting backwards. Two more ways to pick out elements:

- **Masking**: index with a *boolean* array (or a condition, which produces one) to keep only the `True` positions.
- **Fancy indexing**: index with a *list/array of integer positions* to pick out exactly those elements, in that order, without a loop.

Both extend naturally to multiple dimensions (first index = row, second = column).


In [19]:
c = np.array([1, 3, 6, 9, 10, 2])

mask = np.array([False, False, True, False, True, False])
print(c[mask])              # array([6, 10])

mask2 = (c < 4) | (c > 8)   # combine conditions with & (and) / | (or)
print(c[mask2])

ind = [1, 3, 4]
print(c[ind])                # fancy indexing: array([3, 9, 10])


[ 6 10]
[ 1  3  9 10  2]
[ 3  9 10]


One can apply mask and fancy indexing in multidimension.   
Remember that first index is row, and second is column.   
Remember how slicing works: `a[start:end:step]`   : 
- Omitting one value goes up to the end of the sequence. 
- Omitting the second "colon" implies step=1.  
- With negative steps you count backward
- Start/step can be either positive or negative indices (but then you count from the end). 

An illustration of indexing in numpy arrays:
![Illustration of `np` indexing](../Figures/numpy_indexing.png)

**Exercise: indexing playground.** 

Given

```python
M = np.arange(12).reshape((3, 4))
```

find, for each of these, both the result *and* which technique (slicing / masking / fancy indexing / a mix) it uses:
- the value at row 0, column 1
- all rows of column 1
- all elements where `M - 3 < 2`
- the first 2 elements of rows 1 and 0 (in that order)
- all columns from index 2 onward, but only for rows whose row-sum is greater than 20


In [20]:
# Exercise: indexing playground
M = np.arange(12).reshape((3, 4))
print(M)
print('value at row 0, col 1', M[0, 1])
print('All rows of column 1', M[:, 1])
print('All elements where M - 3 < 2', M[M-3 < 2])
print('FIrst 2 elements of rows 1 and 0', M[1::-1, 0:2])
print('all col from 2+, w. row sum > 2', M[M.sum(axis=1) > 20, 2:])

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
value at row 0, col 1 1
All rows of column 1 [1 5 9]
All elements where M - 3 < 2 [0 1 2 3 4]
FIrst 2 elements of rows 1 and 0 [[4 5]
 [0 1]]
all col from 2+, w. row sum > 2 [[ 6  7]
 [10 11]]


In [21]:
# We can dissect the last exercise
# Columns from index 2 onwards: 
print(M[:,2:])
# Now If I want only the rows that have a sum > 20 I need a mask like the following: 
print(M.sum(axis=1) > 20 )
# indeed 
print('sum for each row ', M.sum(axis=1) )

[[ 2  3]
 [ 6  7]
 [10 11]]
[False  True  True]
sum for each row  [ 6 22 38]


In [22]:
# we can also dissect the other (simpler) mask 
print('All elements where M - 3 < 2', M[M-3 < 2])

All elements where M - 3 < 2 [0 1 2 3 4]


In [23]:
M-3 < 2

array([[ True,  True,  True,  True],
       [ True, False, False, False],
       [False, False, False, False]])

## II.5 Reading arrays from a file, and string formatting <a class="anchor" id="II.5"></a>

The simplest way to read/write a plain 2D array from/to a text file: `np.loadtxt('myfile.txt')` and `np.savetxt('myfile.txt', array)`.

**Example.** `data.txt` contains a list of sources: name (`Name`), identifier (`ID`), coordinates (`RA`, `DEC`), estimated redshift (`z`), its uncertainty (`z_err`), and a data-quality flag (`zQF`, where `0` = reliable). Suppose you only want objects with non-zero `z` and `zQF == 0`.


In [24]:
data = np.loadtxt('data.txt')
data

array([[ 1.10000000e+01,  1.10000000e+01,  6.96039800e+01,
        -1.23348300e+01,  1.45026997e-01,  5.39539924e-05,
         0.00000000e+00],
       [ 3.80000000e+01,  3.80000000e+01,  6.95969000e+01,
        -1.23204700e+01,  5.02938603e-01,  2.22303791e-04,
         1.00000000e+00],
       [ 4.20000000e+01,  4.20000000e+01,  6.95713600e+01,
        -1.23179500e+01,  0.00000000e+00,  1.00000000e-03,
         0.00000000e+00],
       [ 5.50000000e+01,  5.50000000e+01,  6.95983200e+01,
        -1.23115700e+01,  0.00000000e+00,  1.00000000e-03,
         0.00000000e+00],
       [ 5.70000000e+01,  5.70000000e+01,  6.95978429e+01,
        -1.23114420e+01,  0.00000000e+00,  1.00000000e-03,
         0.00000000e+00],
       [ 7.20000000e+01,  7.20000000e+01,  6.96111100e+01,
        -1.23037000e+01,  0.00000000e+00,  1.00000000e-03,
         0.00000000e+00],
       [ 8.00000000e+01,  8.00000000e+01,  6.95502300e+01,
        -1.23033900e+01,  0.00000000e+00,  1.00000000e-03,
         0.0000000

**Exercise.** Using masking (see above), keep only the rows with non-zero `z` and `zQF == 0`, then save the trimmed array with `np.savetxt('data_trim.txt', data_trim)`.


In [25]:
# Exercise: filter and save data_trim
data_trim = data[data[:, 6] ==0]
data_trim[0:3]
np.savetxt('data_trim.txt', data_trim)

`np.loadtxt` assumes a simple, uniform, all-numeric array. For anything richer (mixed types, missing values, column names, other formats like csv or fits), a good next step is `astropy.table.Table`, which reads column names directly from the header and prints nicely in a notebook.


In [26]:
from astropy.table import Table

data_tab = Table.read('data.txt', format='ascii')   # specify the format explicitly to avoid guessing errors
data_tab


Name,ID,RA,DEC,z,z_err,zQF
int64,float64,float64,float64,float64,float64,float64
11,11.0,69.60398,-12.33483,0.14502699731,5.39539923983e-05,0.0
38,38.0,69.5969,-12.32047,0.502938602945,0.000222303791245,1.0
42,42.0,69.57136,-12.31795,0.0,0.001,0.0
55,55.0,69.59832,-12.31157,0.0,0.001,0.0
57,57.0,69.5978429,-12.311442,0.0,0.001,0.0
72,72.0,69.61111,-12.3037,0.0,0.001,0.0
80,80.0,69.55023,-12.30339,0.0,0.001,0.0
83,83.0,69.58752,-12.30232,0.0,0.001,0.0
85,85.0,69.56567,-12.30147,0.0,0.001,0.0


In [27]:
print(data_tab['z'][0:3])          # access a column by name, then slice it
z_array = np.array(data_tab['z'])  # cast a column to a plain numpy array if needed


      z       
--------------
 0.14502699731
0.502938602945
           0.0


### String formatting

Two options you'll see constantly, both worth knowing:

**`%`-style (a bit old-school).** The variable(s) come after the string, in a tuple preceded by `%`. Inside the string, `%s` = string, `%d`/`%i` = integer, `%.<n>f` = float with `n` digits after the decimal point, `%.<n>e` = scientific notation.
```python
print('%i is the square root of %i' % (2, 4))
```

**f-strings (modern, since Python 3.6, generally the most readable).** Prefix the string with `f`, and put variables directly inside `{ }`, with an optional format spec after `:`.
```python
print(f'{2} is the square root of {4}')
print(f'{3.14159:.2f}')     # -> '3.14'
```

Some alternative string formating exist. If you do not understand a string formating in a code, just search the web. 

In [28]:
x, y = 2, 4
print('%i is the square root of %i' % (x, y))
print(f'{x} is the square root of {y}')


2 is the square root of 4
2 is the square root of 4


**Exercise.** Create three float variables `a=2.3`, `b=3`, `c=-5`. Print the sentence `a=2.30, b=3 and c=-5.00e+00`, once using `%` formatting, once using an f-string.


In [29]:
# Exercise: a=2.30, b=3 and c=-5.00e+00, with % and with f-strings
a, b, c = 2.3, 3, -5
print('a=%.2f, b=%.i, c=%.2e'%(a, b, c))
print(f'a={a:.2f}, b={b:d}, c={c:.2e}') # formatting is a bit different with f-string

a=2.30, b=3, c=-5.00e+00
a=2.30, b=3, c=-5.00e+00


## II.7 Summary <a class="anchor" id="II.7"></a>

What you need to get started:

- Create arrays: `np.array`, `np.arange`, `np.ones`, `np.zeros`, `np.linspace`.
- Know an array's shape (`.shape`), reshape it (`.reshape()`), flatten it (`.flatten()`).
- `.copy()` for an independent array, `.view()` to share memory, plain `=` for an alias (no new array at all).
- Get a subset (or modify in place) with **masks** (`a[a < 0] = 0`) and **fancy indexing** (`a[[1, 3, 4]]`).
- Aggregate with `.mean()`, `.max()`, `.sum()`, ...; combine differently-shaped arrays via **broadcasting**.
- Read/write plain arrays with `np.loadtxt()`/`np.savetxt()`; for richer/labeled data, `astropy.table.Table`.
- Format numbers for printing with `%` or f-strings.


## References and supplementary material <a class="anchor" id="refs"></a>

- Good video introducing numpy (and that inspired part of this notebook) by J. Vanderplas: https://www.youtube.com/watch?v=EEUXKG97YRw
- Numpy quick-start: https://numpy.org/doc/stable/user/quickstart.html
- Broadcasting: https://docs.scipy.org/doc/numpy/user/basics.broadcasting.html
- String formatting: https://docs.python.org/3/tutorial/inputoutput.html
- `astropy.table.Table`: https://docs.astropy.org/en/stable/table/
